# The harness: shadow mode, status, adapt, fine-tune, then cascade

The journey from "the LLM answers everything" to "Laya answers what it is sure about":

1. `mode="shadow"`: the teacher answers, Laya runs silently and every decision is logged.
2. `h.status()`: per field, how often Laya agrees and how accurate it is when sure.
3. `h.adapt()`: calibrate Laya's confidence and pick safe thresholds (seconds, no training).
4. `h.finetune()`: head-only training on the logged answers; the harness switches only if it passes go/no-go and beats the current student on its held-out test split.
5. `mode={"team": "cascade", ...}`: move a field once the numbers say so.

**Teacher:** this notebook uses a stand-in teacher that answers from the synthetic labels, the way a perfect LLM
would, so it runs without a key and costs nothing. Set `DS_TEACHER=claude-haiku-4-5` (or any LLM) to use a real one.
The student is the real Laya English checkpoint.

Runs on: a laptop (CPU or MPS), a few minutes. No API key with the stand-in teacher.

Built on [Laya](https://github.com/NandhaKishorM/laya) (Apache-2.0) by Nandakishor M / Convai Innovations. decisionsmith is an independent project, not affiliated with TypeSafe AI or Convai Innovations.

In [1]:
# uv pip install "decisionsmith[laya]"
import os

import decisionsmith as ds

from typing import Annotated, Literal

from pydantic import BaseModel, Field


class Ticket(BaseModel):
    team: Annotated[
        Literal["billing", "technical", "sales"],
        ds.Options(billing="payments, invoices, refunds", technical="bugs, outages, errors", sales="pricing, plans"),
    ] = Field(description="Which team should handle this ticket?")
    wants_refund: bool = Field(description="Does the customer ask for their money back?")

In [2]:
import random

TEAMS = {
    "billing": [
        "I was charged twice this month", "my invoice shows the wrong amount", "the tax on my bill looks wrong",
        "there is a fee on my statement I don't recognise", "I need a copy of last month's invoice",
        "my card was charged after I cancelled", "the payment failed but the money left my account",
        "why did my bill go up", "please update the billing address on my invoices", "I was billed for two seats",
    ],
    "technical": [
        "the dashboard shows a 500 error", "I can't log in since the update", "exports are stuck at 99 percent",
        "password reset emails never arrive", "uploads time out after a minute", "the app crashes when I open reports",
        "sync between devices stopped working", "the API returns 401 with a valid key", "search results are empty",
        "the page is blank in Safari",
    ],
    "sales": [
        "can I get a quote for 50 seats", "do you offer discounts for nonprofits", "is there an annual pricing option",
        "what's the difference between pro and team", "I want to upgrade my plan", "can we talk about volume pricing",
        "do you have an enterprise plan", "is there a free trial for teams", "can I pay by invoice for a yearly plan",
        "who do I talk to about a partnership",
    ],
}
OPENERS = ["", "Hi, ", "Hello team, ", "Good morning. ", "Hey, "]
CLOSERS = ["", " Thanks.", " Any update?", " This is urgent.", " Can you help?", " Please look into it."]
REFUNDS = [" I want my money back.", " Please refund me.", " Refund this now!"]


def make(cores, n, seed):
    rng = random.Random(seed)
    rows = []
    for _ in range(n):
        team = rng.choice(sorted(cores))
        core = rng.choice(cores[team])
        refund = team != "sales" and rng.random() < 0.35
        text = rng.choice(OPENERS) + core + "." + rng.choice(REFUNDS if refund else CLOSERS)
        rows.append({"text": text[0].upper() + text[1:], "team": team, "wants_refund": refund})
    return rows


# later traffic uses phrasings never seen before: the last 3 of every team
traffic = make({t: xs[:-3] for t, xs in TEAMS.items()}, 800, seed=0)
later = make({t: xs[-3:] for t, xs in TEAMS.items()}, 200, seed=1)
print(len(traffic), "tickets now,", len(later), "later, e.g.", traffic[0])

800 tickets now, 200 later, e.g. {'text': 'Good morning. do you have an enterprise plan.', 'team': 'sales', 'wants_refund': False}


In [3]:
labels = {r["text"]: {"team": r["team"], "wants_refund": r["wants_refund"]} for r in traffic + later}
TEACHER = os.environ.get("DS_TEACHER") or ds.testing.FakeEngine(labels, confidence=1.0, name="stand-in-llm")
if os.path.exists("decisions.db"):
    os.remove("decisions.db")
h = ds.harness(Ticket, teacher=TEACHER, student="laya", mode="shadow", log="decisions.db")
print(h.decide(traffic[0]["text"]))

<venv>/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<venv>/lib/python3.12/site-packages/laya/agent.py:928: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


id='b57da79f2efe4dd6af42603f822b26a4' value=Ticket(team='sales', wants_refund=False) source={'team': 'teacher', 'wants_refund': 'teacher'} confidence={'team': 1.0, 'wants_refund': 1.0} probabilities={'team': {'billing': 0.0, 'technical': 0.0, 'sales': 1.0}, 'wants_refund': {'false': 1.0, 'true': 0.0}} sure=True latency_ms=3868.2041250285693


The `RuntimeWarning` above comes from Laya: one confidence temperature shipped with the checkpoint is out of range, so Laya clamps it to 0.5 and says confidence from that entry is uncalibrated. decisionsmith fits its own calibration on your data (`train()`, `adapt()`).

## 1. Shadow mode on today's traffic

In [4]:
h.many([r["text"] for r in traffic])
print(h.status())

team:         shadow · student agrees 84% · sure on 57% · accuracy when sure 100% (vs teacher) · 801 labelled -> ready to finetune: run h.finetune()
wants_refund: shadow · student agrees 90% · sure on 81% · accuracy when sure 96% (vs teacher) · 801 labelled -> ready for cascade


## 2. Adapt: calibration and thresholds, no training

In [5]:
print(h.adapt())

adapt: Ticket (student laya)

field         rows  temperature  threshold  coverage  accuracy_when_sure  ece_before  ece_after  confused                        
------------  ----  -----------  ---------  --------  ------------------  ----------  ---------  --------------------------------
team          801   0.500        0.684      0.757     0.981               0.115       0.060      billing/sales (96 of 127 errors)
wants_refund  801   0.821        0.853      0.816     0.971               0.062       0.059      false/true (79 of 79 errors)    
  - thresholds target 97% accuracy on held-out rows; no threshold means the field always asks the teacher


## 3. Fine-tune on the log
The teacher's logged answers are the labels. Head-only below 1,000 rows. The harness switches to the new checkpoint only if it passes go/no-go and beats the current student on its held-out test split (logged rows it did not train on).

In [6]:
report = h.finetune(out="runs/tickets")
print(report)
print("student now:", h.student.name)

loading laya on mps


base accuracy on test: 0.900  (240 decisions)
training the head on 601 rows (fresh, loss=ce)


epoch 1/4  train loss 0.2408  calib loss 0.1486  (63s)


epoch 2/4  train loss 0.2105  calib loss 0.1421  (72s)


epoch 3/4  train loss 0.1830  calib loss 0.1102  (73s)


epoch 4/4  train loss 0.1659  calib loss 0.1092  (92s)


finetune: laya -> runs/tickets

field         test  base_acc  accuracy  macro_f1  base_ece  ece  
------------  ----  --------  --------  --------  --------  -----
team          120   0.883     0.900     0.904     0.130     0.054
wants_refund  120   0.917     1.000     1.000     0.066     0.005
all           240   0.900     0.950     0.934     0.076     0.026

go: yes
  - the harness now uses student='laya:runs/tickets'; pass that next time you build it
saved: ./runs/tickets
student now: laya:runs/tickets


## 4. Keep measuring on new traffic (new phrasings), then read the status again

In [7]:
h.many([r["text"] for r in later])
print(h.adapt())
print(h.status())

adapt: Ticket (student laya:runs/tickets)

field         rows  temperature  threshold  coverage  accuracy_when_sure  ece_before  ece_after  confused                       
------------  ----  -----------  ---------  --------  ------------------  ----------  ---------  -------------------------------
team          200   5.000        -          0.000     -                   0.213       0.322      billing/sales (25 of 37 errors)
wants_refund  200   1.562        0.737      1.000     0.990               0.009       0.009      false/true (2 of 2 errors)     
  - thresholds target 97% accuracy on held-out rows; no threshold means the field always asks the teacher
team:         shadow · student agrees 82% · sure on 0% · 1001 labelled -> ready to finetune: run h.finetune()
wants_refund: shadow · student agrees 99% · sure on 100% · accuracy when sure 99% (vs teacher) · 1001 labelled -> ready for cascade


**What this run showed:** the fine-tune passed go/no-go on held-out logged rows (0.900 -> 0.950), but for `team` it did not carry over to the new phrasings: `adapt()` set its temperature to 5.0 with no threshold, and the student is sure on 0% of them. So `team` stays in shadow and the teacher keeps answering it; that is what the teacher fallback and the audit are for. `wants_refund` is ready for cascade.

## 5. Move the fields that are ready to cascade
Only fields `status()` calls ready move; the rest stay in shadow. In cascade the student answers when it is sure; otherwise the teacher does, and 5% of sure answers are still checked by the teacher (`audit`). Cascaded answers can still be wrong, which is why a share of them is audited.

In [8]:
ready = [n for n, f in h.status().fields.items() if "ready for cascade" in f.advice]
modes = {n: "cascade" if n in ready else "shadow" for n in h.schema.fields}
print(modes)
h2 = ds.harness(Ticket, teacher=TEACHER, student=h.student, mode=modes, log="decisions.db")
for r in later[:8]:
    d = h2.decide(r["text"])
    print(d.value, d.source, r["text"])

{'team': 'shadow', 'wants_refund': 'cascade'}
team='billing' wants_refund=False {'team': 'teacher', 'wants_refund': 'student'} I was billed for two seats. Any update?
team='billing' wants_refund=False {'team': 'teacher', 'wants_refund': 'student'} Good morning. please update the billing address on my invoices. Please look into it.


team='sales' wants_refund=False {'team': 'teacher', 'wants_refund': 'student'} Is there a free trial for teams. This is urgent.
team='billing' wants_refund=False {'team': 'teacher', 'wants_refund': 'student'} Please update the billing address on my invoices. Please look into it.
team='sales' wants_refund=False {'team': 'teacher', 'wants_refund': 'student'} Hi, can I pay by invoice for a yearly plan. Can you help?
team='billing' wants_refund=True {'team': 'teacher', 'wants_refund': 'teacher'} Please update the billing address on my invoices. Refund this now!


team='technical' wants_refund=False {'team': 'teacher', 'wants_refund': 'student'} Good morning. the API returns 401 with a valid key. Please look into it.
team='billing' wants_refund=False {'team': 'teacher', 'wants_refund': 'student'} Hey, please update the billing address on my invoices. Thanks.
